- Compare six arms on the recorded ten-galaxy set.
- Use retrieved posterior medians and fitted-pixel pulls.
- Band pull = sum((data − model)/effective_sigma)/sqrt(n).
- Band edges: [Trager et al. 1998, Table 2](https://arxiv.org/pdf/astro-ph/9712258#page=37).
- Use existing air-to-vacuum conversion and target redshifts.
- Empty bands remain missing. Record fitted and native pixels.
- Age derivatives use fixed-alpha posterior medians, Gyr/dex.
- Slopes describe these ten targets, with no selection change.
- Fit seed 20260832. Bootstrap seed 20261110.

In [1]:
from pathlib import Path
import json
import sys
import h5py
import numpy as np
import pandas as pd
from scipy.stats import linregress

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
sys.path.insert(0, str(ROOT))
from scripts.experiment import validate_result
from ceridwen.observation.absorption_features import feature_windows

OUT = ROOT / 'results/afe-ladder-floor-2026-11-10'
manifest = json.loads((OUT / 'run/manifest.json').read_text())
cells = manifest['source']['experiment']
assert set(manifest['completed_cells']) == {c['name'] for c in cells}
FEATURES = ['Mgb', 'C4668', 'CN4150', 'G4300', 'Fe4383']
rows = []
for cell in cells:
    directory = OUT / 'run/fits' / cell['arm'] / f"{cell['target_metadata']['object_id']}-{cell['target']}"
    validate_result(directory)
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        quantiles = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ['q16', 'q50', 'q84']}
        row = dict(arm=cell['arm'], target=cell['target'], z=float(f.attrs['redshift']),
                   source_commit=manifest['source']['commit'], seed=int(f.attrs['random_seed']))
        parameters = {'age_gyr': 'mass-weighted age [Gyr]', 'feh': 'logzsol',
                      'tau_dust': 'diffuse_tau_noll', 'afe': 'afe', 'f_calib': 'log_f_calib'}
        for label, parameter in parameters.items():
            for q in quantiles:
                value = quantiles[q][parameter] if parameter in quantiles[q] else float(cell['settings']['fix_afe'])
                row[f'{label}_{q}'] = float(np.exp(value) if label == 'f_calib' else value)
        diag = f['diagnostics'].attrs
        row.update(lnZ=float(diag['log_evidence']), lnZ_err=float(diag['log_evidence_err']),
                   diagnostics_passed=bool(diag['passed']))
        spectrum = f['spectrum']
        wave = spectrum['wavelength'][:]
        pulls = spectrum['pull'][:]
        fitted = spectrum['mask'][:].astype(bool) & np.isfinite(pulls)
        for name in FEATURES:
            lo, hi = feature_windows([name], zred=row['z'])[0]
            native = (wave >= lo) & (wave <= hi)
            use = native & fitted
            n = int(use.sum())
            row[f'{name}_n'] = n
            row[f'{name}_native_n'] = int(native.sum())
            row[f'{name}_range_covered'] = bool(wave[0] <= lo and wave[-1] >= hi)
            row[f'{name}_pull'] = float(pulls[use].sum() / np.sqrt(n)) if n else np.nan
        rows.append(row)
per_galaxy = pd.DataFrame(rows)
assert len(per_galaxy) == 60 and per_galaxy.diagnostics_passed.all()
base = per_galaxy[per_galaxy.arm == 'free_afe'].set_index('target')
METRICS = ['age_gyr_q50', 'feh_q50', 'tau_dust_q50', 'f_calib_q50', 'afe_q50', 'lnZ'] + [f'{f}_pull' for f in FEATURES]
for metric in METRICS:
    per_galaxy[f'delta_{metric}'] = per_galaxy[metric] - per_galaxy.target.map(base[metric])
per_galaxy.to_csv(OUT / 'per-galaxy.csv', index=False)
summary = per_galaxy.groupby('arm', sort=False)[METRICS + [f'delta_{m}' for m in METRICS]].median()
summary['galaxies'] = per_galaxy.groupby('arm', sort=False).size()
for name in FEATURES:
    summary[f'{name}_galaxies'] = per_galaxy.groupby('arm', sort=False)[f'{name}_pull'].count()
summary.to_csv(OUT / 'arm-medians.csv')
display(summary[['galaxies', 'age_gyr_q50', 'feh_q50', 'tau_dust_q50', 'f_calib_q50', 'afe_q50', 'lnZ']])
summary[[f'{name}_pull' for name in FEATURES] + [f'{name}_galaxies' for name in FEATURES]]

,galaxies,age_gyr_q50,feh_q50,tau_dust_q50,f_calib_q50,afe_q50,lnZ
arm,,,,,,,
free_afe,10,4.213187,-0.268764,0.440213,0.031426,-0.021594,235552.131177
afe_0p0,10,4.124183,-0.288525,0.435181,0.031421,0.000000,235552.787443
afe_0p2,10,4.215257,-0.378891,0.417528,0.032019,0.200000,235528.145718
afe_0p4,10,4.448493,-0.562738,0.401353,0.033188,0.400000,235475.309723
data_floor_free,10,4.168167,-0.255004,0.454804,0.030554,-0.013921,235557.049955
data_floor_afe_0p2,10,4.246155,-0.374091,0.426186,0.030895,0.200000,235530.771561


,Mgb_pull,C4668_pull,CN4150_pull,G4300_pull,Fe4383_pull,Mgb_galaxies,C4668_galaxies,CN4150_galaxies,G4300_galaxies,Fe4383_galaxies
arm,,,,,,,,,,
free_afe,-4.088035,-3.690433,-1.725536,1.200165,-0.348758,5,9,10,10,10
afe_0p0,-4.785768,-3.621742,-1.586347,1.200239,-0.322245,5,9,10,10,10
afe_0p2,-2.950846,-4.995215,-2.176250,1.212106,-2.262901,5,9,10,10,10
afe_0p4,-2.144304,-5.526723,-2.331238,0.993502,-3.588497,5,9,10,10,10
data_floor_free,-3.729241,-3.461730,-1.572232,1.366717,-0.115896,5,9,10,10,10
data_floor_afe_0p2,-2.854923,-4.361903,-2.068289,1.397678,-2.057131,5,9,10,10,10


- Fixed-alpha differences retain each galaxy as its own control.

In [2]:
ages = per_galaxy.pivot(index='target', columns='arm', values='age_gyr_q50')
sensitivity = pd.DataFrame(index=ages.index)
sensitivity['age_afe0p0_gyr'] = ages.afe_0p0
sensitivity['age_afe0p2_gyr'] = ages.afe_0p2
sensitivity['age_afe0p4_gyr'] = ages.afe_0p4
sensitivity['dage_dafe_0p0_to_0p2_gyr_per_dex'] = (ages.afe_0p2 - ages.afe_0p0) / 0.2
sensitivity['dage_dafe_0p2_to_0p4_gyr_per_dex'] = (ages.afe_0p4 - ages.afe_0p2) / 0.2
sensitivity['dage_dafe_ladder_gyr_per_dex'] = (ages.afe_0p4 - ages.afe_0p0) / 0.4
sensitivity.loc['median'] = sensitivity.median()
sensitivity.to_csv(OUT / 'age-sensitivity.csv')
sensitivity

,age_afe0p0_gyr,age_afe0p2_gyr,age_afe0p4_gyr,dage_dafe_0p0_to_0p2_gyr_per_dex,dage_dafe_0p2_to_0p4_gyr_per_dex,dage_dafe_ladder_gyr_per_dex
target,,,,,,
M14_37219,3.404916,3.327904,3.437851,-0.385061,0.549737,0.082338
M1_210210,3.211980,3.926766,5.348375,3.573925,7.108049,5.340987
M2_134021,3.315181,3.321076,3.616725,0.029474,1.478242,0.753858
M3_117010,4.307081,4.283461,4.530127,-0.118098,1.233326,0.557614
M3_120308,4.419409,4.616683,5.412275,0.986369,3.977960,2.482165
M4_107370,4.220611,4.214179,4.324694,-0.032158,0.552575,0.260209
M5_238314,4.115228,4.216335,4.366859,0.505531,0.752622,0.629076
M7_146213,4.133137,4.681091,5.271215,2.739769,2.950619,2.845194
M8_150848,3.543059,3.993212,4.334337,2.250769,1.705623,1.978196


- Resample galaxies together across arms for descriptive age-redshift slopes.

In [3]:
rng = np.random.default_rng(20261110)
indices = rng.integers(0, len(base), size=(10000, len(base)))
z = base.z.to_numpy()
z_draws = z[indices]
z_centered = z_draws - z_draws.mean(axis=1, keepdims=True)
denominator = np.sum(z_centered**2, axis=1)
slope_rows = []
for arm, group in per_galaxy.groupby('arm', sort=False):
    y = group.set_index('target').loc[base.index, 'age_gyr_q50'].to_numpy()
    fit = linregress(z, y)
    with np.errstate(invalid='ignore', divide='ignore'):
        draws = np.sum(z_centered * y[indices], axis=1) / denominator
    q16, q50, q84 = np.nanquantile(draws, [0.16, 0.5, 0.84])
    slope_rows.append(dict(arm=arm, galaxies=len(y), z_min=z.min(), z_max=z.max(),
        slope_gyr_per_z=fit.slope, bootstrap_q16=q16, bootstrap_q50=q50, bootstrap_q84=q84,
        residual_scatter_gyr=np.std(y - (fit.intercept + fit.slope*z), ddof=2),
        bootstrap_seed=20261110, bootstrap_draws=10000))
slopes = pd.DataFrame(slope_rows)
slopes.to_csv(OUT / 'age-redshift-slopes.csv', index=False)
slopes

,arm,galaxies,z_min,z_max,slope_gyr_per_z,bootstrap_q16,bootstrap_q50,bootstrap_q84,residual_scatter_gyr,bootstrap_seed,bootstrap_draws
0,free_afe,10,0.6542,0.891105,1.275206,-0.438609,1.227784,2.612809,0.480424,20261110,10000
1,afe_0p0,10,0.6542,0.891105,2.019854,0.318073,1.973523,3.419417,0.461378,20261110,10000
2,afe_0p2,10,0.6542,0.891105,0.390423,-1.436115,0.332364,1.338398,0.496844,20261110,10000
3,afe_0p4,10,0.6542,0.891105,-2.282378,-5.828140,-2.440290,-0.440531,0.697688,20261110,10000
4,data_floor_free,10,0.6542,0.891105,1.178906,-0.725165,1.120803,2.737754,0.495223,20261110,10000
5,data_floor_afe_0p2,10,0.6542,0.891105,0.058650,-1.796213,0.006899,1.011489,0.517304,20261110,10000


- Noticed, not done: C/N responses, mocks, full 135-galaxy chronometer test.
- Proposal: `notebooks/ceridwen_integrated_photometry_spectra.ipynb`, “Corners and SFH” cell: `KeyError: afe`. Export fixed-value draws when α/Fe is fixed.
- Recovery changes result copies only. Original failed notebooks remain.
- `recovery.json` records unchanged sample hashes and validation.
- Noticed, not done: state inspection references an absent historical worktree.